<center>
<h1><b> TP3: Prompt Engineering </b>
<h1><b>Classification tasks : Sentiment Analysis </b>
</center>

**Étudiant :** ELAALGI Hatim — Module Système Distribué Agentique, Master S3

# Objectives

- Present LLMs as an effective tool to tackle classification problems.
- Illustrate prompt engineering for sentiment analysis.
- Build and evaluate LLMs on broader classification tasks.

## Mes choix pour ce TP

| Élément | Notebook / PDF du prof | Ce notebook |
|---|---|---|
| Tokens | `tiktoken`, encodage `o200k_base` | identique |
| Appel LLM | OpenAI `gpt-4o`, ou Ollama `qwen3.5:cloud` | **Groq** (API gratuite) : `qwen/qwen3.8-27b`, de la même famille Qwen que le modèle Ollama du prof |
| Données | IMDB (Hugging Face) | `dataset_imdb.zip` fourni (50 000 critiques) |
| Gold examples | 20, `random_state=42`, découpage 80 / 20 | identique |
| Few-shot | 4 positifs + 4 négatifs mélangés | identique |
| Évaluation | micro-F1, 10 répétitions | identique, sur **deux modèles** : un petit (`allam-2-7b`, 7 milliards de paramètres) et un plus grand (`qwen/qwen3.8-27b`) |

Pourquoi deux modèles ? Dans le PDF, `gpt-5.2` obtient 1.0 avec les trois prompts : un modèle aussi fort ne permet pas de voir ce qu'apportent le few-shot et le CoT. Avec un petit modèle, l'effet du prompt devient mesurable.

La clé est lue dans le fichier `.env` à la racine du dépôt (`GROQ_API_KEY=...`), qui n'est pas versionné.

# Preparer l'environement

1. Prendre VS Code, je vais créer un un nouveau dossier que je vais appeler **SMA_TokPr** et ouvrir le terminal.
2. **UV** install: utiliser la la commande, pour MacOS: ```curl -LsSf https://astral.sh/uv/install.sh | sh```  sous Windows utiliser la commande dans le powerShell: `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"`. Après instalation de UV quittez et relancez VScode.
3. dans le terminal de VsCode utiliser la commande ` uv init ` pour initialiser le projet. Donc ça nous permet de générer la structure des projets notamment avec le fichier **.toml** .
4. créer un environnement virtuel. Avec la commande: ` uv venv ` .
3. autoriser PowerSell à exécuter les scripts avec cette commande dans le terminal: ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` . puis aller au dosssier **Scripts** et activer les scripts avec la commande:  ` .\.venv\Scripts\activate `.
5.  cree un notebook **prompttoken.ipynb** pour le test puis choisir un noyaux(le kernel) et tester avec `print("Bonjour")` .

7. installer la bib tiktoken avec ` uv add tiktoken `

> **Dans mon dépôt**, l'environnement uv est celui du conteneur Docker du TP0. J'ai ajouté les dépendances du TP3 au même projet (`uv add tiktoken langchain langchain-groq python-dotenv scikit-learn tqdm`), puis reconstruit l'image (`docker compose build`). Le kernel **Python (uv-env)** du conteneur exécute ce notebook.

In [ ]:
print("Bonjour")

# Partie 1 — Les tokens

In [ ]:
import tiktoken

In [ ]:
tokenizer=tiktoken.encoding_for_model("gpt-4o")

In [ ]:
print(tokenizer.name)

In [ ]:
tokenizer=tiktoken.get_encoding("o200k_base")

In [ ]:
prompt="Vous êtes un expert en marketing digital. C'est quoi un digital marketer ? "

In [ ]:
tokens=tokenizer.encode(prompt)

In [ ]:
print(tokens)

In [ ]:
print(len(tokens))

In [ ]:
for token in tokens:
    t=tokenizer.decode_single_token_bytes(token).decode("utf-8")
    print(t,end='|')

**Ce qu'on observe.** Le tokeniseur ne découpe ni par lettre ni par mot : il découpe en **morceaux fréquents** de son vocabulaire (`o200k_base`, environ 200 000 entrées, celui de GPT-4o). L'espace est collé au début du mot suivant (`| expert`), et la ponctuation forme ses propres tokens. Le modèle ne voit que la liste d'entiers : c'est elle qu'il faut compter pour estimer le coût d'un appel et vérifier qu'un prompt tient dans la fenêtre de contexte.

**Test personnel : la même phrase en français, en anglais et en arabe.**

In [ ]:
phrases = {
    "français": "Je suis étudiant en master d'intelligence artificielle à Tanger.",
    "anglais": "I am a master's student in artificial intelligence in Tangier.",
    "arabe": "أنا طالب ماستر في الذكاء الاصطناعي في طنجة.",
}
for langue, phrase in phrases.items():
    ids = tokenizer.encode(phrase)
    morceaux = [tokenizer.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in ids]
    print(f"{langue:<9} {len(phrase.split()):>2} mots -> {len(ids):>2} tokens : {'|'.join(morceaux)}")

@@TOKENS@@

# Utilisation des llm

8. installer la dépendance langchain avec la commande: ` uv add langchain ` et installer langchain-openai ` uv add langchain-openai `

**Code du prof (OpenAI)**, que je remplace par Groq :
```python
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o", temperature=0)
```
Avec LangChain, changer de fournisseur revient à changer de classe : `ChatGroq` (paquet `langchain-groq`) a la même interface que `ChatOpenAI`.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from IPython.display import Markdown, display

load_dotenv(override=True)   # cherche le fichier .env en remontant les dossiers
print("GROQ_API_KEY trouvée :", bool(os.getenv("GROQ_API_KEY")))

In [ ]:
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

**apel** a llm

In [ ]:
resp=llm.invoke(input=[{"role":"system","content":"Vous êtes un expert en marketing digital"},{"role":"user","content":"C'est quoi un digital marketer ?"}])

In [ ]:
display(Markdown(resp.content))

In [ ]:
# Ce que l'appel a coûté, en tokens (lien avec la partie 1)
usage = resp.response_metadata["token_usage"]
print("tokens du prompt :", usage["prompt_tokens"], "| tokens de la réponse :", usage["completion_tokens"])
print("tiktoken (o200k_base) compte", len(tokenizer.encode("Vous êtes un expert en marketing digital" + "C'est quoi un digital marketer ?")),
      "tokens pour les deux messages")

**Remarques.**
- Le message `system` fixe le rôle (« expert en marketing digital ») et le message `user` porte la question. La réponse est un `AIMessage` dont le texte est dans `.content`.
- `temperature=0` choisit toujours le token le plus probable : la réponse est (presque) **déterministe**, ce dont on a besoin pour comparer des prompts dans la partie 3.
- Le nombre de tokens facturés par Groq est plus élevé que le compte de `tiktoken`, pour deux raisons : Qwen a **son propre tokeniseur**, et le modèle ajoute des tokens de formatage autour de chaque message (rôles, début et fin de message).
- Dans le notebook du prof, `print(display(...))` affiche aussi `None`, car `display()` ne renvoie rien. J'appelle donc `display()` seul.

# ollama llm

Le prof propose aussi un modèle via **Ollama** (`ollama run qwen3.5:cloud`, puis `uv add langchain-ollama`) :
```python
from langchain_ollama import ChatOllama
llm2=ChatOllama(model="qwen3.5:cloud", temperature=0)
resp2=llm2.invoke(input=[{"role":"system","content":"Vous êtes un expert en marketing digital"},{"role":"user","content":"C'est quoi un digital marketer ?"}])
```
Je n'ai pas installé Ollama sur ma machine : le modèle Qwen utilisé plus haut via Groq joue ce rôle. Le code est le même ; seule la classe change (`ChatOllama` au lieu de `ChatGroq`).

---
# Partie 2 — Use case : Sentiment Analysis

## Step 1 : Define Objectives and Metrics
Chaque critique de film reçoit l'une des deux étiquettes **positive** ou **negative**. On mesure :
- l'**accuracy** : la part de prédictions correctes ;
- le **micro-F1** : on additionne les TP, FP et FN de toutes les classes, puis on calcule $F_1 = \frac{2PR}{P+R}$.

Je refais le calcul de l'exemple du PDF (TP = 50, FP = 10, FN = 5, TN = 100) avec `sklearn` :

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

# Matrice du PDF : 50 TP, 10 FP, 5 FN, 100 TN  ->  165 exemples
y_true = ["positive"] * 50 + ["negative"] * 10 + ["positive"] * 5 + ["negative"] * 100
y_pred = ["positive"] * 50 + ["positive"] * 10 + ["negative"] * 5 + ["negative"] * 100

print("F1 de la classe 'positive' :", round(f1_score(y_true, y_pred, pos_label="positive"), 3))
print("micro-F1 (2 classes)        :", round(f1_score(y_true, y_pred, average="micro"), 3))
print("accuracy                    :", round(accuracy_score(y_true, y_pred), 3))

**Remarque sur l'exemple du PDF.** La valeur 0,870 du PDF est le F1 de la **seule classe positive** ($2 \cdot 50 / (2 \cdot 50 + 10 + 5)$). Le **micro-F1** sur les deux classes compte aussi les 100 négatifs bien classés comme vrais positifs de la classe *negative*. Il vaut 0,909, c'est-à-dire exactement l'accuracy (150 / 165). Ce n'est pas un hasard : quand chaque exemple reçoit **une seule** étiquette, chaque erreur compte une fois en FP (classe prédite) et une fois en FN (vraie classe), donc micro-précision = micro-rappel = accuracy. Dans la suite, `f1_score(..., average="micro")` est donc égal à l'accuracy, y compris quand le modèle répond `unknown`.

## Step 2 : Assemble Data
Le PDF charge IMDB avec `load_dataset("imdb")`. J'utilise l'archive `dataset_imdb.zip` fournie, que `pandas` lit directement sans la décompresser.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split

train_df = pd.read_csv("dataset_imdb.zip")
train_df.info()

In [ ]:
train_df.sentiment.value_counts()

In [ ]:
train_df.sentiment.hist()
plt.title("Répartition des étiquettes")
plt.show()

**Prétraitement.**
- La colonne texte s'appelle `review` : je la renomme `text`, comme dans le PDF.
- Le PDF ajoute `sentiment` à partir de `label`. Ici, c'est l'inverse : le CSV a déjà `sentiment`, et j'en déduis `label` (1 = positive).
- Je retire les balises `<br />` laissées par le scraping.
- **Je garde les critiques de 80 mots au plus.** C'est un écart par rapport au PDF, imposé par deux contraintes. `allam-2-7b` a une fenêtre de contexte de **4 096 tokens**, et un prompt few-shot contient 8 critiques en plus de la critique à classer : avec des critiques longues (jusqu'à 2 000 mots dans IMDB), il déborderait. De plus, l'offre gratuite de Groq limite les tokens par minute et par jour.

In [ ]:
MAX_MOTS = 80

train_df = train_df.rename(columns={"review": "text"})
train_df["text"] = train_df["text"].str.replace(r"<br\s*/?>", " ", regex=True).str.strip()
train_df["label"] = np.where(train_df.sentiment == "positive", 1, 0)
train_df = train_df[train_df["text"].str.split().str.len() <= MAX_MOTS].reset_index(drop=True)

print(len(train_df), "critiques de", MAX_MOTS, "mots au plus")
print(train_df.sentiment.value_counts())
train_df.sample(6, random_state=7)

Découpage 80 % / 20 % : un réservoir d'**exemples** few-shot et un réservoir de **gold examples** pour l'évaluation (code du PDF).

In [ ]:
examples_df, gold_examples_df = train_test_split(
    train_df, test_size=0.2, random_state=42
)
(examples_df.shape, gold_examples_df.shape)

In [ ]:
columns= ['text', 'sentiment']
gold_examples = (
    gold_examples_df.loc[:, columns]
    .sample(20, random_state=42)
    .to_json(orient='records'))

# Le jeu curé est enregistré en JSON pour les itérations suivantes
os.makedirs("data", exist_ok=True)
with open("data/gold_examples.json", "w", encoding="utf-8") as f:
    f.write(gold_examples)

json.loads(gold_examples)[0]

In [ ]:
pd.Series([e["sentiment"] for e in json.loads(gold_examples)]).value_counts()

## Step 3 : Élaborer le prompt

### Prompt 1 : Zero-shot

In [ ]:
zero_shot_system_message = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'
Do not explain your answer.
"""

In [ ]:
zero_shot_prompt = [{'role':'system', 'content': zero_shot_system_message}]

In [ ]:
user_message_template = """```{movie_review}```"""

### Prompt 2 : Few-shot
Même message système, enrichi d'exemples. Pour éviter le **biais de l'étiquette majoritaire**, on prend autant de positifs que de négatifs ; pour éviter le **biais de récence**, on les mélange. J'ajoute un paramètre `random_state` à la fonction du PDF : chaque tirage reste aléatoire, mais il est reproductible d'une exécution à l'autre.

In [ ]:
few_shot_system_message = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'
Do not explain your answer.
"""


def create_examples(dataset, n=4, random_state=None):
    positive_reviews = (dataset.sentiment == 'positive')
    negative_reviews = (dataset.sentiment == 'negative')
    columns_to_select = ['text', 'sentiment']
    positive_examples = dataset.loc[positive_reviews, columns_to_select].sample(n, random_state=random_state)
    negative_examples = dataset.loc[negative_reviews, columns_to_select].sample(n, random_state=random_state)
    examples = pd.concat([positive_examples, negative_examples])
    # sampling without replacement is equivalent to random shuffling
    randomized_examples = examples.sample(2*n, replace=False, random_state=random_state)
    return randomized_examples.to_json(orient='records')


examples = create_examples(examples_df, 2, random_state=0)
json.loads(examples)

In [ ]:
def create_prompt(system_message, examples, user_message_template):
    few_shot_prompt = [{'role':'system', 'content': system_message}]
    for example in json.loads(examples):
        example_review = example['text']
        example_sentiment = example['sentiment']
        few_shot_prompt.append(
            {'role': 'user', 'content': user_message_template.format(movie_review=example_review)}
        )
        few_shot_prompt.append(
            {'role': 'assistant', 'content': f"{example_sentiment}"}
        )
    return few_shot_prompt


examples = create_examples(examples_df, random_state=42)   # n=4 -> 8 exemples
few_shot_prompt = create_prompt(few_shot_system_message, examples, user_message_template)
[m["role"] for m in few_shot_prompt]

### Prompt 3 : Chain-of-Thought
Mêmes exemples, mais le message système ajoute des instructions étape par étape.

In [ ]:
cot_system_message = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks ``` in the input.
Answer only 'positive' or 'negative'
Do not explain your answer.

Instructions:
1. Carefully read the text of the review and think through the options for sentiment provided
2. Consider the overall sentiment of the review and estimate the probability of the review being positive

To reiterate, your answer should strictly only contain the label: positive or negative
"""

cot_few_shot_prompt = create_prompt(cot_system_message, examples, user_message_template)

Taille des trois prompts, mesurée avec `tiktoken` (partie 1), avant d'y ajouter la critique à classer :

In [ ]:
def nb_tokens(messages):
    return sum(len(tokenizer.encode(m["content"])) for m in messages)

for nom, p in [("zero-shot", zero_shot_prompt), ("few-shot", few_shot_prompt), ("CoT few-shot", cot_few_shot_prompt)]:
    print(f"{nom:<13} {len(p):>2} messages, ~{nb_tokens(p)} tokens")

## Step 4 : Evaluate Prompts

### Les deux modèles évalués

In [ ]:
petit_modele = ChatGroq(model="allam-2-7b", temperature=0, max_retries=0)
grand_modele = ChatGroq(model="qwen/qwen3.8-27b", temperature=0, max_retries=0)

**Appels à l'API.** L'évaluation complète représente environ 1 000 appels. Pour les rendre robustes, `appeler()` ajoute deux choses au simple `model.invoke()` :
- un **cache** sur disque (`data/cache_llm.json`) : une requête identique (même modèle, mêmes messages) n'est jamais envoyée deux fois. Si l'exécution est interrompue, elle reprend là où elle s'était arrêtée, et les résultats restent reproductibles sans clé ;
- une **attente** quand Groq répond « 429 : limite de tokens par minute atteinte » : on attend le délai indiqué par l'API, puis on réessaie. Si c'est le quota **journalier** qui est épuisé, on s'arrête avec un message clair.

In [ ]:
import hashlib
import time
from groq import RateLimitError

CACHE_PATH = "data/cache_llm.json"
cache = json.load(open(CACHE_PATH, encoding="utf-8")) if os.path.exists(CACHE_PATH) else {}


class QuotaJournalierAtteint(Exception):
    pass


def appeler(model, messages):
    cle = hashlib.sha256(json.dumps([model.model_name, messages], ensure_ascii=False).encode()).hexdigest()
    if cle not in cache:
        for essai in range(20):
            try:
                cache[cle] = model.invoke(messages).content
                break
            except RateLimitError as e:
                attente = float(e.response.headers.get("retry-after", 10))
                if attente > 600:
                    raise QuotaJournalierAtteint(f"{model.model_name} : réessayer dans {attente / 3600:.1f} h") from e
                time.sleep(attente + 1)
        with open(CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
    return cache[cle]

`evaluate_prompt` est la fonction du PDF. Seul l'appel `model.invoke(prompt+user_input)` passe par `appeler()`. Je garde aussi les prédictions (`derniers_resultats`) pour pouvoir analyser les erreurs ensuite.

In [ ]:
derniers_resultats = {}


def evaluate_prompt(prompt, gold_examples, user_message_template, model):
    model_predictions, ground_truths = [], []

    for example in json.loads(gold_examples):
        gold_input = example['text']
        user_input = [
            {
                'role':'user',
                'content': user_message_template.format(movie_review=gold_input)
            }
        ]
        try:
            response_content = appeler(model, prompt+user_input)
            if 'negative' in response_content.strip().lower():
                prediction='negative'
            elif 'positive' in response_content.strip().lower():
                prediction='positive'
            else:
                prediction ='unknown'
            model_predictions.append(prediction)
            ground_truths.append(example['sentiment'])
        except QuotaJournalierAtteint:
            raise
        except Exception as e:
            print(e)
            continue

    derniers_resultats["predictions"], derniers_resultats["truths"] = model_predictions, ground_truths
    micro_f1_score = f1_score(ground_truths, model_predictions, average="micro")
    return micro_f1_score

### Évaluation unique des trois prompts, sur les deux modèles

In [ ]:
prompts = {"zero-shot": zero_shot_prompt, "few-shot": few_shot_prompt, "CoT few-shot": cot_few_shot_prompt}
modeles = {"allam-2-7b": petit_modele, "qwen3.8-27b": grand_modele}

scores, predictions = {}, {}
for nom_modele, modele in modeles.items():
    for nom_prompt, prompt in prompts.items():
        scores[(nom_modele, nom_prompt)] = evaluate_prompt(prompt, gold_examples, user_message_template, modele)
        predictions[(nom_modele, nom_prompt)] = derniers_resultats["predictions"]

tableau = pd.Series(scores).unstack().loc[list(modeles), list(prompts)]
tableau.round(2)

Quelles critiques sont mal classées, et par qui ?

In [ ]:
gold_list = json.loads(gold_examples)
lignes = []
for k, ex in enumerate(gold_list):
    preds = {f"{m} / {p}": predictions[(m, p)][k] for (m, p) in predictions}
    if any(v != ex["sentiment"] for v in preds.values()):
        lignes.append({"vérité": ex["sentiment"], **preds, "critique": ex["text"]})

erreurs = pd.DataFrame(lignes)
print(len(erreurs), "critiques gold sur 20 sont mal classées par au moins une combinaison modèle / prompt")
pd.set_option("display.max_colwidth", None)
erreurs

Réponses brutes du petit modèle qui ne sont ni `positive` ni `negative` (prédiction `unknown`) :

In [ ]:
for k, ex in enumerate(gold_list):
    for nom_prompt, prompt in prompts.items():
        if predictions[("allam-2-7b", nom_prompt)][k] == "unknown":
            msg = prompt + [{"role": "user", "content": user_message_template.format(movie_review=ex["text"])}]
            print(f"[{nom_prompt}] critique {k} :", repr(appeler(petit_modele, msg))[:200])

### Évaluations répétées : 10 tirages d'exemples

Les prompts few-shot et CoT dépendent des 8 exemples tirés. On répète donc l'évaluation 10 fois, avec un nouveau tirage à chaque fois, sur le petit modèle (celui où le prompt compte).

**Correction de la boucle du PDF.** Dans le PDF, la dernière ligne est `cot_few_shot_performance.append(few_shot_micro_f1)` : la liste CoT reçoit le score **few-shot**, et le prompt CoT n'est jamais évalué. C'est pour cela que le PDF obtient exactement la même moyenne pour les deux. Ici, le prompt CoT est construit avec les **mêmes** exemples que le few-shot de la même répétition, puis évalué séparément.

In [ ]:
num_eval_runs = 10
few_shot_performance, cot_few_shot_performance = [], []
for run in tqdm(range(num_eval_runs)):
    # For each run create a new sample of examples
    examples = create_examples(examples_df, random_state=run)
    # Assemble the few shot prompt with these examples
    few_shot_prompt = create_prompt(few_shot_system_message, examples, user_message_template)
    cot_few_shot_prompt = create_prompt(cot_system_message, examples, user_message_template)
    # Evaluate prompt accuracy on gold examples
    few_shot_micro_f1 = evaluate_prompt(few_shot_prompt, gold_examples, user_message_template, petit_modele)
    cot_few_shot_micro_f1 = evaluate_prompt(cot_few_shot_prompt, gold_examples, user_message_template, petit_modele)
    few_shot_performance.append(few_shot_micro_f1)
    cot_few_shot_performance.append(cot_few_shot_micro_f1)

In [ ]:
for nom, perf in [("few-shot", few_shot_performance), ("CoT few-shot", cot_few_shot_performance)]:
    mean_val = np.array(perf).mean()
    std_val = np.array(perf).std()
    print(f"{nom:<13} f1_score Moyen= {mean_val:.3f}, Standard Deviation = {std_val:.3f}   runs = {np.round(perf, 2).tolist()}")

In [ ]:
zero_shot_petit = scores[("allam-2-7b", "zero-shot")]
plt.figure(figsize=(7, 4.5))
plt.boxplot([few_shot_performance, cot_few_shot_performance], tick_labels=["few-shot", "CoT few-shot"])
rng = np.random.default_rng(0)
for i, perf in enumerate([few_shot_performance, cot_few_shot_performance], start=1):
    plt.scatter(i + rng.uniform(-0.07, 0.07, len(perf)), perf, alpha=0.7)
plt.axhline(zero_shot_petit, color="gray", linestyle="--", label=f"zero-shot = {zero_shot_petit:.2f}")
plt.ylabel("micro-F1 sur les 20 gold examples")
plt.title("allam-2-7b : variabilité sur 10 tirages d'exemples")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
print("Appels distincts enregistrés dans le cache :", len(cache))

@@ANALYSE@@